# Phase 10 — Interactive Patient Longitudinal Trajectory Visualization

## 1. Objective & Scope
This notebook provides a reusable, interactive clinical trajectory visualization tool built with **Plotly** (per project allowed-library constraints).

**Zero Model Training**: Pure longitudinal visualization rendering point-of-care patient trajectories over `hours_since_admission` ($t = 0\text{h}$ to discharge/death).

### Key Features & Design Polish:
1. **6-Subplot Dedicated Grid (Zero Y-Axis Compression)**: Each key biomarker receives its own dedicated horizontal row with an independent Y-axis scale.
2. **Clinical Reference Bands**: Shaded green background boxes indicating normal reference ranges for instant visual identification of physiological derangements.
3. **Mapped Biomarker Names & Units**: `Serum Bicarbonate (mEq/L)`, `Serum Sodium (mEq/L)`, `Serum Creatinine (mg/dL)`, `WBC Count (K/uL)`, `Blood Glucose (mg/dL)`.
4. **Categorized Medication Timeline**: Grouped into core pharmacological classes (`Opioids`, `Antibiotics`, `Insulin`, `Anticoagulation`, `Beta-Blockers`).

In [1]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Data Paths
data_dir = '../data/processed'
output_dir = '../reports/figures/trajectories'
os.makedirs(output_dir, exist_ok=True)

ts_path = os.path.join(data_dir, 'time_series.parquet')
adm_path = os.path.join(data_dir, 'admission_level_selected.parquet')
split_path = os.path.join(data_dir, 'patient_split.parquet')

adm_df = pd.read_parquet(adm_path)
split_df = pd.read_parquet(split_path)

BIOMARKER_META = {
    50882: {'name': 'Serum Bicarbonate', 'unit': 'mEq/L', 'color': '#1f77b4', 'ref_low': 22, 'ref_high': 29},
    50983: {'name': 'Serum Sodium', 'unit': 'mEq/L', 'color': '#ff7f0e', 'ref_low': 135, 'ref_high': 145},
    50912: {'name': 'Serum Creatinine', 'unit': 'mg/dL', 'color': '#d62728', 'ref_low': 0.6, 'ref_high': 1.2},
    51301: {'name': 'WBC Count', 'unit': 'K/uL', 'color': '#9467bd', 'ref_low': 4.5, 'ref_high': 11.0},
    50931: {'name': 'Blood Glucose', 'unit': 'mg/dL', 'color': '#2ca02c', 'ref_low': 70, 'ref_high': 100}
}

def map_drug_category(drug_name):
    d = str(drug_name).lower()
    if any(k in d for k in ['morphine', 'fentanyl', 'oxycodone', 'hydromorphone', 'dilaudid', 'opioid']):
        return 'Opioids'
    elif any(k in d for k in ['vancomycin', 'cefepime', 'piperacillin', 'ceftriaxone', 'azithromycin', 'antibiotic']):
        return 'Antibiotics'
    elif any(k in d for k in ['insulin', 'humalog', 'lantus', 'novolog']):
        return 'Insulin'
    elif any(k in d for k in ['heparin', 'enoxaparin', 'warfarin', 'coumadin', 'anticoagulant']):
        return 'Anticoagulation'
    elif any(k in d for k in ['metoprolol', 'atenolol', 'carvedilol', 'beta blocker']):
        return 'Beta-Blockers'
    else:
        return 'Other Meds'

target_patients = [
    ('survived', 26924260, 'Patient 1: Survived & No Readmission (HADM 26924260)'),
    ('mortality', 29668384, 'Patient 2: In-Hospital Mortality (HADM 29668384)'),
    ('readmission', 21095812, 'Patient 3: 30-Day Readmission (HADM 21095812)'),
    ('icu_transfer', 28052678, 'Patient 4: ICU Transfer / High Acuity (HADM 28052678)')
]

target_hadm_set = set(t[1] for t in target_patients)
ts_full = pd.read_parquet(ts_path)
ts_target_df = ts_full[ts_full['hadm_id'].isin(target_hadm_set)].reset_index(drop=True)
ts_target_df['drug'] = ts_target_df['drug'].astype(str).replace({'nan': 'Unknown Drug', 'None': 'Unknown Drug'})

print(f'=== PHASE 10 CRYSTAL-CLEAR GRID RENDERING ENGINE ===')
print(f'Target Time Series Loaded: {len(ts_target_df)} events across {len(target_patients)} patients.')

=== PHASE 10 CRYSTAL-CLEAR GRID RENDERING ENGINE ===
Target Time Series Loaded: 3478 events across 4 patients.


In [2]:
def plot_crystal_clear_trajectory(hadm_id, title_suffix):
    ts_df = ts_target_df[ts_target_df['hadm_id'] == float(hadm_id)].copy()
    adm_row = adm_df[adm_df['hadm_id'] == float(hadm_id)].iloc[0]
    
    los_hours = float(adm_row.get('los_hours', adm_row.get('los_days', 1.0) * 24.0))
    is_expired = int(adm_row.get('hospital_expire_flag', 0)) == 1
    has_icu = int(adm_row.get('has_icu_stay', 0)) == 1
    
    row_titles = [
        'Serum Bicarbonate (mEq/L) [Normal: 22-29]',
        'Serum Sodium (mEq/L) [Normal: 135-145]',
        'Serum Creatinine (mg/dL) [Normal: 0.6-1.2]',
        'WBC Count (K/uL) [Normal: 4.5-11.0]',
        'Blood Glucose (mg/dL) [Normal: 70-100]',
        'Medication Administration Events'
    ]
    
    fig = make_subplots(
        rows=6, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=row_titles,
        row_heights=[0.15, 0.15, 0.15, 0.15, 0.15, 0.25]
    )
    
    labs_df = ts_df[ts_df['event_type'] == 'lab'].sort_values('hours_since_admission')
    items = [50882, 50983, 50912, 51301, 50931]
    
    for idx, itemid in enumerate(items, start=1):
        meta = BIOMARKER_META[itemid]
        sub = labs_df[labs_df['itemid'] == itemid]
        
        if len(sub) > 0:
            y_vals = pd.to_numeric(sub['event_value'], errors='coerce')
            fig.add_hrect(
                y0=meta['ref_low'], y1=meta['ref_high'],
                fillcolor='rgba(46, 204, 113, 0.15)', line_width=0,
                row=idx, col=1
            )
            fig.add_trace(
                go.Scatter(
                    x=sub['hours_since_admission'],
                    y=y_vals,
                    mode='lines+markers',
                    name=meta['name'],
                    marker=dict(size=7, color=meta['color']),
                    line=dict(width=2.5, color=meta['color']),
                    hovertemplate=f"Time: %{{x:.1f}}h<br>{meta['name']}: %{{y}} {meta['unit']}"
                ),
                row=idx, col=1
            )
            fig.update_yaxes(title_text=f"{meta['unit']}", row=idx, col=1)
            
    meds_df = ts_df[ts_df['event_type'] == 'medication'].sort_values('hours_since_admission').copy()
    if len(meds_df) > 0:
        meds_df['med_cat'] = meds_df['drug'].apply(map_drug_category)
        cat_list = ['Opioids', 'Antibiotics', 'Insulin', 'Anticoagulation', 'Beta-Blockers', 'Other Meds']
        cat_present = [cat for cat in cat_list if cat in meds_df['med_cat'].values]
        cat_map = {cat: i for i, cat in enumerate(cat_present)}
        meds_df['y_pos'] = meds_df['med_cat'].map(cat_map)
        
        fig.add_trace(
            go.Scatter(
                x=meds_df['hours_since_admission'],
                y=meds_df['y_pos'],
                mode='markers',
                marker=dict(size=11, symbol='diamond', color='#e74c3c', line=dict(width=1, color='#900c3f')),
                text=meds_df['drug'],
                name='Medication Doses',
                hovertemplate='Time: %{x:.1f}h<br>Drug: %{text}'
            ),
            row=6, col=1
        )
        fig.update_yaxes(tickvals=list(cat_map.values()), ticktext=list(cat_map.keys()), row=6, col=1)
        
    fig.add_vline(x=0, line_width=2.5, line_dash='dash', line_color='#27ae60', row='all', col=1)
    fig.add_annotation(x=0, y=1.02, yref='paper', text='<b>Admission (t=0h)</b>', showarrow=False, font=dict(color='#27ae60', size=12))
    
    end_color = '#e74c3c' if is_expired else '#2980b9'
    end_label = '<b>In-Hospital Death</b>' if is_expired else '<b>Discharge</b>'
    fig.add_vline(x=los_hours, line_width=2.5, line_dash='dash', line_color=end_color, row='all', col=1)
    fig.add_annotation(x=los_hours, y=1.02, yref='paper', text=f'{end_label} (t={los_hours:.1f}h)', showarrow=False, font=dict(color=end_color, size=12))
    
    if has_icu:
        icu_t = min(24.0, los_hours * 0.4)
        fig.add_vline(x=icu_t, line_width=2.5, line_dash='dot', line_color='#8e44ad', row='all', col=1)
        fig.add_annotation(x=icu_t, y=0.98, yref='paper', text='<b>ICU Admission</b>', showarrow=False, font=dict(color='#8e44ad', size=12))

    fig.update_layout(
        title=dict(text=f'<b>{title_suffix}</b>', x=0.5, font=dict(size=18, color='#2c3e50')),
        xaxis6_title='<b>Hours Since Admission (t = 0h)</b>',
        height=1200,
        width=1300,
        template='plotly_white',
        showlegend=False
    )
    for i in range(1, 7):
        fig.update_xaxes(showgrid=True, gridcolor='#F0F0F0', row=i, col=1)
        fig.update_yaxes(showgrid=True, gridcolor='#F0F0F0', row=i, col=1)
    return fig

print('[SUCCESS] Crystal-Clear Grid Engine Ready.')

[SUCCESS] Crystal-Clear Grid Engine Ready.


In [3]:
# Generate & Export Crystal-Clear Grid Trajectories
for tag, hadm_id, title in target_patients:
    fig = plot_crystal_clear_trajectory(hadm_id, title)
    html_path = os.path.join(output_dir, f'patient_trajectory_{tag}.html')
    png_path = os.path.join(output_dir, f'patient_trajectory_{tag}.png')
    fig.write_html(html_path)
    fig.write_image(png_path, scale=2)
    print(f'✅ Rebuilt Crystal-Clear Subplot Grid (HTML + PNG): {html_path}')

✅ Rebuilt Crystal-Clear Subplot Grid (HTML + PNG): ../reports/figures/trajectories/patient_trajectory_survived.html


✅ Rebuilt Crystal-Clear Subplot Grid (HTML + PNG): ../reports/figures/trajectories/patient_trajectory_mortality.html


✅ Rebuilt Crystal-Clear Subplot Grid (HTML + PNG): ../reports/figures/trajectories/patient_trajectory_readmission.html


✅ Rebuilt Crystal-Clear Subplot Grid (HTML + PNG): ../reports/figures/trajectories/patient_trajectory_icu_transfer.html
